# Train SMD Component Detector & Export TFLite

7791 images, 4 classes. One click: Run → 15 min → download .tflite

In [ ]:
!pip install -q roboflow ultralytics onnx
print("OK")

In [ ]:
from roboflow import Roboflow
import os, yaml, shutil

rf = Roboflow(api_key="aLtYgQ8E7uKE7s3FNO20")
project = rf.workspace("dainius").project("smdcomponents")
dataset = project.version(6).download("yolov5")
print(f"Root: {dataset.location}")

# Fix nested folder structure
yaml_path = None
for root, dirs, files in os.walk(dataset.location):
    for f in files:
        if f == "data.yaml":
            yaml_path = os.path.join(root, f)
            break

print(f"YAML: {yaml_path}")
print(f"  Exists: {os.path.exists(yaml_path)}")

# Read & fix paths in data.yaml
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

base = os.path.dirname(yaml_path)
print(f"  Base: {base}")

# Check if train/val paths point to correct location
cfg['path'] = base
cfg['train'] = os.path.join(base, 'train', 'images')
cfg['val'] = os.path.join(base, 'valid', 'images')
cfg['test'] = os.path.join(base, 'test', 'images') if os.path.exists(os.path.join(base, 'test')) else ''

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print(f"  Train: {cfg['train']} → exists: {os.path.exists(cfg['train'])}")
print(f"  Val:   {cfg['val']} → exists: {os.path.exists(cfg['val'])}")

In [ ]:
!sed -i 's|path:.*|path: {base}|' {yaml_path} 2>/dev/null || true

!yolo detect train \
  data={yaml_path} \
  model=yolov5nu.pt \
  epochs=20 \
  imgsz=416 \
  batch=32 \
  name=smd_detector \
  exist_ok=True \
  device=0

print("Training done!")

In [ ]:
from ultralytics import YOLO
import glob, os

pt_files = glob.glob("runs/detect/smd_detector/**/best.pt", recursive=True)
pt = pt_files[0]
print(f"Loading {pt}")

model = YOLO(pt)
model.export(format="tflite", imgsz=416)

tfl = glob.glob("runs/detect/smd_detector/**/*.tflite", recursive=True)
print(f"TFLite: {tfl[0]}")

from google.colab import files
files.download(tfl[0])